# Week 7 — RAG Preparation Notebook (Sandbox)

This notebook is a sandbox environment only. Its purpose is to test and tune the parsing and chunking logic and metadata extraction, before we move any finalized code into the production src/rag/week7/*.py files.

Each section below has a specific test, and each cell is isolated from the others (especially the reading cell from the chunking cell) so we can try different chunking values dozens of times without re-reading the files from disk each time.

**Sections:**
1. Text extraction (Parsing) test — `.txt` and `.pdf`
2. Metadata extraction test from the filename
3. Chunking algorithm engineering and tuning (Chunking Tuning)
4. Simulating the final row format ready for storage in SQLite

## Section 0 — Setup

Note: Currently, all the week7 files we have are `.txt` / `.md` only, while the next section will test pypdf on an actual PDF file. So, the cell below generates a realistic PDF version of the invoice_2024-01 invoice only if it does not already exist (including a Service Description / Quantity / Unit Price table) so we can test the "table trap" mentioned in the engineering notes on it.

This file is temporary for experimentation in the notebook.

In [1]:
import os
from pathlib import Path

BASE_DIR = Path("../../data/raw/week7")
INVOICES_DIR = BASE_DIR / "invoices"
POLICIES_DIR = BASE_DIR / "policies"
REPORTS_DIR = BASE_DIR / "reports"

pdf_path = INVOICES_DIR / "invoice_2024-01_projectAlpha_computeEngine.pdf"

if not pdf_path.exists():
    from fpdf import FPDF

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "CloudTech Solutions - Invoice", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", size=11)
    pdf.ln(2)
    pdf.cell(0, 8, "Date: January 15, 2024", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 8, "Project: Alpha", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 8, "Service: Compute Engine", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)
    pdf.multi_cell(0, 7, "This invoice covers the compute resources utilized for Project Alpha "
                          "during the month of January 2024.")
    pdf.ln(4)

    col_widths = [90, 40, 40]
    pdf.set_font("Helvetica", "B", 11)
    for w, h in zip(col_widths, ["Service Description", "Quantity", "Unit Price"]):
        pdf.cell(w, 8, h, border=1)
    pdf.ln()

    pdf.set_font("Helvetica", size=11)
    rows = [
        ("High-Memory Instance (n2-highmem-8)", "500 hrs", "$8.50"),
        ("Persistent SSD Storage", "1200 GB", "$0.17"),
        ("Network Egress (Data Pipeline)", "300 GB", "$0.12"),
    ]
    for r in rows:
        for w, val in zip(col_widths, r):
            pdf.cell(w, 8, val, border=1)
        pdf.ln()

    pdf.ln(6)
    pdf.set_font("Helvetica", "B", 12)
    pdf.cell(0, 8, "Total amount due: $4,500 USD", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", size=11)
    pdf.cell(0, 8, "Please process the payment within 30 days to avoid service interruption.",
              new_x="LMARGIN", new_y="NEXT")

    pdf.output(str(pdf_path))
    print(f"[setup] generated test PDF at: {pdf_path}")
else:
    print(f"[setup] PDF already exists, skipping generation: {pdf_path}")


[setup] PDF already exists, skipping generation: ..\..\data\raw\week7\invoices\invoice_2024-01_projectAlpha_computeEngine.pdf


## Section 1 — Parsing Experiment

In [2]:
with open(REPORTS_DIR / "report_2024-03_projectBeta_security.txt", "r", encoding="utf-8") as f:
    txt_raw_text = f.read()

print(txt_raw_text)

Report Title: Security Vulnerability Scan
Date: March 05, 2024
Project: Beta
Service: Security Operations

A routine security scan was conducted on Project Beta's staging environment in March 2024.
Two low-level vulnerabilities were detected regarding outdated SSL certificates on the frontend servers.
Action taken: The certificates were successfully rotated and updated on March 6th. The system is now fully compliant with our internal security standards.


## 1.b) Reading a PDF file using the pypdf library


Here we will closely monitor how the table (Service Description / Quantity / Unit Price) is read.

In [3]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
pdf_raw_text = ""
for page in reader.pages:
    pdf_raw_text += page.extract_text()

print(pdf_raw_text)


CloudTech Solutions - Invoice
Date: January 15, 2024
Project: Alpha
Service: Compute Engine
This invoice covers the compute resources utilized for Project Alpha during the month of January 2024.
Service Description Quantity Unit Price
High-Memory Instance (n2-highmem-8) 500 hrs $8.50
Persistent SSD Storage 1200 GB $0.17
Network Egress (Data Pipeline) 300 GB $0.12
Total amount due: $4,500 USD
Please process the payment within 30 days to avoid service interruption.


## Section 2 — Metadata Extraction Test
An experimental function that takes the filename and splits it (split('_')) to extract: document type, date, project, and service.

**Essential requirement:** The function must not crash if the extension is `.txt` instead of `.pdf` (or `.md`) — extraction depends only on the filename, not its content type.

In [4]:
import os
import re


def extract_metadata_from_filename(filename: str) -> dict:
    """Parse a rag document filename following the convention:
    <type>_<YYYY-MM>_<project>_<service>.<ext>
    Robust to a missing service segment (defaults to 'general') and to
    any file extension (.pdf / .txt / .md ...).
    """
    base, ext = os.path.splitext(filename)
    ext = ext.lstrip(".").lower()
    parts = base.split("_")

    metadata = {
        "document_name": filename,
        "source_format": ext,
        "document_type": None,
        "month": None,
        "project": None,
        "service": None,
    }

    if len(parts) < 3:
        print(f"[WARN] filename '{filename}' does not match the naming convention, "
              f"partial metadata only")
        return metadata

    metadata["document_type"] = parts[0]

    if re.match(r"^\d{4}-\d{2}$", parts[1]):
        metadata["month"] = parts[1]
    else:
        print(f"[WARN] '{parts[1]}' in '{filename}' is not a valid YYYY-MM month")

    metadata["project"] = parts[2]
    
    metadata["service"] = parts[3] if len(parts) > 3 else "general"

    return metadata


In [5]:
test_filenames = [
    "invoice_2024-01_projectAlpha_computeEngine.pdf",
    "invoice_2024-01_projectAlpha_computeEngine.txt",
    "policy_2024-01_projectAlpha_general.md", 
    "report_2024-03_projectBeta_security.txt",
    "weird_file_name.txt",                              
]

for name in test_filenames:
    print(name, "->", extract_metadata_from_filename(name))


invoice_2024-01_projectAlpha_computeEngine.pdf -> {'document_name': 'invoice_2024-01_projectAlpha_computeEngine.pdf', 'source_format': 'pdf', 'document_type': 'invoice', 'month': '2024-01', 'project': 'projectAlpha', 'service': 'computeEngine'}
invoice_2024-01_projectAlpha_computeEngine.txt -> {'document_name': 'invoice_2024-01_projectAlpha_computeEngine.txt', 'source_format': 'txt', 'document_type': 'invoice', 'month': '2024-01', 'project': 'projectAlpha', 'service': 'computeEngine'}
policy_2024-01_projectAlpha_general.md -> {'document_name': 'policy_2024-01_projectAlpha_general.md', 'source_format': 'md', 'document_type': 'policy', 'month': '2024-01', 'project': 'projectAlpha', 'service': 'general'}
report_2024-03_projectBeta_security.txt -> {'document_name': 'report_2024-03_projectBeta_security.txt', 'source_format': 'txt', 'document_type': 'report', 'month': '2024-03', 'project': 'projectBeta', 'service': 'security'}
[WARN] 'file' in 'weird_file_name.txt' is not a valid YYYY-MM mon

## Section 3 — Chunking Engineering and Tuning

**Step 1 — Initial (Naive/"Blind") Algorithm:** Chunking every 300 characters with a 50-character overlap, without any awareness of word or sentence boundaries.

In [6]:
def naive_chunk(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """Blind slicing: cuts at a fixed character count regardless of word/sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

In [7]:
with open(POLICIES_DIR / "policy_2024-01_projectAlpha_general.md", "r", encoding="utf-8") as f:
    policy_raw_text = f.read()

naive_chunks = naive_chunk(policy_raw_text, chunk_size=300, overlap=50)
for i, c in enumerate(naive_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(repr(c))
    print()


--- chunk 0 (300 chars) ---
'# General Access Policy\n**Project:** Alpha\n**Effective Date:** January 1, 2024\n\n## 1. Authentication\nAll data engineers working on Project Alpha must use Multi-Factor Authentication (MFA) to access the production SQLite database. \nPasswords must be at least 16 characters long and rotated every 90 da'

--- chunk 1 (257 chars) ---
't least 16 characters long and rotated every 90 days.\n\n## 2. Remote Access\nAccessing the compute engines from outside the corporate network requires a secure VPN connection. No direct SSH access is permitted from public IP addresses under any circumstances.'

--- chunk 2 (7 chars) ---
'tances.'



**Visual Inspection:** Notice that the first chunk ends with `...rotated every 90 day` — the word `days` was cut in half and became just `day`! This is exactly the "Blind Chunking" we must avoid, because it makes the chunk lose its meaning and later negatively impacts retrieval quality in RAG.

**Step 2 — Optimization: Cutting at the nearest Whitespace Boundary**
Instead of cutting exactly at character 300, we go back to the nearest whitespace before it and cut there. This is

In [8]:
def smart_chunk(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """Cuts at the nearest whitespace before the target boundary, so words are never split in half."""
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = min(start + chunk_size, text_len)
        if end < text_len:
            boundary = text.rfind(" ", start, end)
            if boundary != -1 and boundary > start:
                end = boundary
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        next_start = end - overlap
        start = next_start if next_start > start else end
    return chunks


In [9]:
smart_chunks = smart_chunk(policy_raw_text, chunk_size=300, overlap=50)
for i, c in enumerate(smart_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(repr(c))
    print()

--- chunk 0 (297 chars) ---
'# General Access Policy\n**Project:** Alpha\n**Effective Date:** January 1, 2024\n\n## 1. Authentication\nAll data engineers working on Project Alpha must use Multi-Factor Authentication (MFA) to access the production SQLite database. \nPasswords must be at least 16 characters long and rotated every 90'

--- chunk 1 (260 chars) ---
'e at least 16 characters long and rotated every 90 days.\n\n## 2. Remote Access\nAccessing the compute engines from outside the corporate network requires a secure VPN connection. No direct SSH access is permitted from public IP addresses under any circumstances.'

--- chunk 2 (49 chars) ---
'from public IP addresses under any circumstances.'



**Visual Inspection (Comparison):** Now the first chunk ends with `...rotated every 90` (it stopped before the word `days` instead of cutting it in half) — a clear improvement over the naive version.

**There is still room for further improvement:** The cut can still happen in the middle of a sentence (not just in the middle of a word). As a next step (outside the scope of this experimental notebook, to be written in the final version in `chunking.py`), a more mature algorithm prefers cutting at sentence-ending punctuation (`. `) or at double line breaks (`\n\n`) that separate paragraphs/headings, before falling back to just the nearest whitespace. We will document this note in `Report_week7.md` as a future improvement, not a current "bug".

## Section 4 — Simulating the Final Data Shape (Data Structure Preview)

Combining the outputs of sections 2 and 3 into a `dict` that approximately represents the row that will later be stored in the `rag_chunks` SQLite table.

In [10]:
invoice_meta = extract_metadata_from_filename("invoice_2024-01_projectAlpha_computeEngine.pdf")

preview_rows = []
for idx, chunk_text in enumerate(smart_chunks):
    row = {
        "document_name": invoice_meta["document_name"],
        "chunk_index": idx,
        "chunk_text": chunk_text,
        "project": invoice_meta["project"],
        "month": invoice_meta["month"],
    }
    preview_rows.append(row)

for row in preview_rows:
    print(row)


{'document_name': 'invoice_2024-01_projectAlpha_computeEngine.pdf', 'chunk_index': 0, 'chunk_text': '# General Access Policy\n**Project:** Alpha\n**Effective Date:** January 1, 2024\n\n## 1. Authentication\nAll data engineers working on Project Alpha must use Multi-Factor Authentication (MFA) to access the production SQLite database. \nPasswords must be at least 16 characters long and rotated every 90', 'project': 'projectAlpha', 'month': '2024-01'}
{'document_name': 'invoice_2024-01_projectAlpha_computeEngine.pdf', 'chunk_index': 1, 'chunk_text': 'e at least 16 characters long and rotated every 90 days.\n\n## 2. Remote Access\nAccessing the compute engines from outside the corporate network requires a secure VPN connection. No direct SSH access is permitted from public IP addresses under any circumstances.', 'project': 'projectAlpha', 'month': '2024-01'}
{'document_name': 'invoice_2024-01_projectAlpha_computeEngine.pdf', 'chunk_index': 2, 'chunk_text': 'from public IP addresses under 

**Note:** This format is the simplest "preview" required for the experiment (only 5 fields: document_name, chunk_index, chunk_text, project, month). In the production script (`src/rag/week7/ingest.py`), we will expand it to also include `document_type`, `service`, `source_format`, and `char_count` — as agreed upon in the `rag_documents` / `rag_chunks` table schemas, but there is no need to complicate the experimental notebook with them right now.

## Summary of Engineering Notes (it will be documented later in `Report_week7.md`)

- **The PDF Tables Trap:** `pypdf` returns sequential text in a correct format but loses the column structure — any accurate future tabular extraction will require another library (`pdfplumber`/`camelot`), not a chunking optimization.
- **Encoding:** `encoding='utf-8'` must be explicitly set in every text file read — essential so the code doesn't break on a different production environment.
- **Blind Chunking:** The naive algorithm (exactly 300 characters) cuts words in the middle. The improved version (whitespace boundary) partially solves this; cutting at sentence/paragraph boundaries is the next planned improvement for the production version.
- **Cell Isolation:** The reading cell (Section 1) is completely separated from the chunking cell (Section 3) — we can experiment with `chunk_size`/`overlap` dozens of times without re-reading the files from the disk.

In [15]:
from pypdf import PdfReader

pdf_path = "E:\\DigitalFutureProjects\\TrainingProjects\\standalone-data-rag-assistant\\data\\raw\\week7\\misc\\الوظيفة الثانية (1).pdf"

reader = PdfReader(pdf_path)
raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text()

print("=== Raw Text Output ===")
print(repr(raw_text))

=== Raw Text Output ===
'الوظيفة الثانية \nنريد تطوير نظام تعليم الكتروني يهدف إلى تسهيل عملية التعليم من خلال إدارة المستخدمين (مسؤولين، مدربين، \nمتعلمين)، نشر الدورات التعليمية، إدارة عمليات الاشتراك والدفع، والتقييمات والاختبارات: \nيمكن للمسؤول admin إضافةمدربين، بحيث يتمكن المدرب من تسجيل الدخول للنظام وإضافة دورات تابعة له \nوإدارتها، يقوم المدرب بإنشاء اختبار لكل دورة لتقييم المشتركين وتتبع حالتهم. \nيجب أن تتم الموافقة من قبل الـ admin  عند إضافة دورة جديدة \nيمكن للمستخدمين التسجيل في النظام واستعراض الدورات المتاحة والاشتراك بها بعد دفع الرسوم الخاص لكل دورة، \nكما يمكنه استعراض جميع الدورات التي اشترك بها وحالته في كل منها.  \nيجب على المشتركين اجتياز الاختيار في نهاية كل دورة. \nالمطلوب: \n1. تصميم بنية الخدمات: \n\uf0b7 اقترح هيكلية نظام مكوّن من عدة خدمات Microservices) تلبي المتطلباتالسابقة. \n2. تنفيذ الـWeb Service: \n\uf0b7 تطبيق المنطق الوظيفي عبر واجهات خدمات ويبREST/soap). \n3. ضمان توجيه جميع طلبات المستخدمين عبر بوابة موحدةAPI Gateway \n4. عنونة الخدمات بشكل دين

In [17]:
import sys, os

def find_project_root(marker='requirements.txt'):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looked for {marker})")

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.append(project_root)

print(project_root)

e:\DigitalFutureProjects\TrainingProjects\standalone-data-rag-assistant


In [20]:
from pypdf import PdfReader
from src.rag.week7.chunking import DocumentChunker
from src.rag.week7.text_cleaner import TextCleaner, strip_repeated_header_footer

def test_pipeline_on_pdf(file_path: str):
    print(f"========== TESTING PIPELINE ON: {file_path} ==========\n")
    
    # 1.(Load Phase)
    reader = PdfReader(file_path)
    raw_pages = [page.extract_text() for page in reader.pages]

    cleaned_pages = strip_repeated_header_footer(raw_pages)
    
    raw_text = "\n".join(cleaned_pages)

    print("=== 1. RAW TEXT (First 300 chars) ===")
    print(repr(raw_text[:300]))
    print("-" * 50)

    # 2.(Clean Phase)
    cleaner = TextCleaner()
    cleaned_text = cleaner.clean(text=raw_text, aggressive=True)

    print("\n=== 2. CLEANED TEXT (Visual Inspection - First 500 chars) ===")
    print(repr(cleaned_text[:500])) 
    print("-" * 50)

    # 3.(Chunk Phase)
    chunker = DocumentChunker(chunk_size=300, overlap=50)
    
    chunks = chunker.chunk(text=cleaned_text, allow_header_presplit=False)

    print(f"\n=== 3. CHUNKING RESULTS (Total: {len(chunks)} chunks) ===")
    for i, chunk_dict in enumerate(chunks[:5]):
        print(f"\n--- Chunk {i} ---")
        print(f"Section Header: {chunk_dict.get('section_header')}")
        print(f"Length: {len(chunk_dict['chunk_text'])} chars")
        print(f"Text: {repr(chunk_dict['chunk_text'])}")


test_pipeline_on_pdf("E:\\DigitalFutureProjects\\TrainingProjects\\standalone-data-rag-assistant\\data\\raw\\week7\\misc\\Task-3.pdf")

========== TESTING PIPELINE ON: E:\DigitalFutureProjects\TrainingProjects\standalone-data-rag-assistant\data\raw\week7\misc\Task-3.pdf ==========

=== 1. RAW TEXT (First 300 chars) ===
'Trainee\n \n3:\n \nConversational\n \nFinOps\n \nAssistant\n \nSpecialist\n \n●\n \nMain\n \nTask:\n \nDevelop\n \nan\n \nAI-powered\n \nconversational\n \nassistant\n \nfor\n \nanswering\n \nFinOps\n \nqueries\n \nin\n \nnatural\n \nlanguage.\n \n●\n \nObjective:\n \nTo\n \nprovide\n \nan\n \nintuitive\n \ninterface\n \nfor\n \nusers\n \nto\n \nget\n \nreal-tim'
--------------------------------------------------

=== 2. CLEANED TEXT (Visual Inspection - First 500 chars) ===
'Trainee 3:\n\n Conversational FinOps Assistant Specialist * Main Task:\n\n Develop an AI-powered conversational assistant for answering FinOps queries in natural language.\n\n * Objective:\n\n To provide an intuitive interface for users to get real-time cost insights without deep technical knowledge.\n\n This trainee will build the